# 02b - 合成数据生成

使用 Moonshot API 从 548 条种子数据生成约 6000 条合成 TRIZ 训练数据。

**预计时间**: 4-8 小时（取决于 Moonshot API 速率限制）
**预计成本**: 约 ￥5-10 元

**执行前准备**:
1. 确保已设置环境变量 `MOONSHOT_API_KEY`
2. 建议先运行 Notebook 01 验证环境
3. 检查 Moonshot API 速率限制 tier（Tier 0 = 3 RPM, Tier 1 = 200 RPM）

## 2b.1 导入依赖

In [ ]:
import sys
import os
import json
from pathlib import Path

sys.path.append('/home/meerkat/mongoose_ai')

# 配置和工具导入
from config import (
    DATA_DIR, DATA_CONFIG, SYNTHETIC_CONFIG,
    BASE_MODEL, MODELS_DIR
)
from utils.data_utils import (
    load_raw_data, convert_to_chatml, split_dataset, save_dataset
)
from utils.synthetic_pipeline import (
    MoonshotSyntheticClient, SyntheticPipeline
)
from utils.pipeline_state import PipelineState
from utils.training_utils import load_model_and_tokenizer

print("依赖导入完成")
print(f"数据目录: {DATA_DIR}")
print(f"合成配置: {SYNTHETIC_CONFIG['api']['model']}")

## 2b.2 加载种子数据

In [ ]:
# 加载种子数据
seed_data = load_raw_data(DATA_CONFIG['raw_data_dir'])

# 如果raw_data_dir为空，从sample_data.json加载
if not seed_data or sum(len(v) for v in seed_data.values()) == 0:
    from utils.data_utils import create_sample_data
    seed_data = create_sample_data()

print(f"已加载 {len(seed_data)} 个子集:")
total_seeds = 0
for name, samples in seed_data.items():
    print(f"  {name}: {len(samples)} 条种子")
    total_seeds += len(samples)
print(f"\n种子总数: {total_seeds}")

# 显示配置中的扩展倍数
print("\n扩展配置:")
for subset, mult in SYNTHETIC_CONFIG['multipliers'].items():
    strategy = SYNTHETIC_CONFIG['strategies'][subset]
    count = len(seed_data.get(subset, []))
    print(f"  {subset}: {count} 种子 x {mult} 倍 = ~{count * mult} 条 ({strategy})")

## 2b.3 初始化 API 客户端并估算成本

In [ ]:
# 初始化 Moonshot API 客户端
api_config = SYNTHETIC_CONFIG['api']

client = MoonshotSyntheticClient(
    api_key=os.environ.get('MOONSHOT_API_KEY'),
    model=api_config['model'],
    rpm=api_config['rpm'],
)

# 估算总成本
print("=" * 60)
print("成本估算")
print("=" * 60)

total_seeds_for_gen = 0
for subset_name, seeds in seed_data.items():
    total_seeds_for_gen += len(seeds)

cost = client.estimate_cost(
    seed_count=total_seeds_for_gen,
    batch_size=api_config['batch_size']
)

print(f"种子总数: {cost['seed_count']}")
print(f"批次大小: {cost['batch_size']}")
print(f"总批次: {cost['num_batches']}")
print(f"预计输入 tokens: {cost['estimated_input_tokens']:,}")
print(f"预计输出 tokens: {cost['estimated_output_tokens']:,}")
print(f"预计成本: ￥{cost['estimated_cost_cny']} (约 ${cost['estimated_cost_usd']})")
print(f"预计时间: {cost['estimated_time_minutes']} 分钟 (RPM={cost['rpm']})")
print("=" * 60)

# 检查 API key
if not os.environ.get('MOONSHOT_API_KEY'):
    print("\n警告: MOONSHOT_API_KEY 环境变量未设置！")
    print("请在继续前设置: export MOONSHOT_API_KEY='your-key'")
else:
    print("\nMOONSHOT_API_KEY 已设置")

## 2b.4 生成合成数据（支持检查点恢复）

此步骤会逐个子集生成合成数据。每个批次完成后自动保存检查点，
如果中断可以重新运行此单元格从中断处继续。

In [ ]:
# 初始化合成流水线
pipeline = SyntheticPipeline(
    client=client,
    output_dir=SYNTHETIC_CONFIG['output_dir'],
    checkpoint_dir=SYNTHETIC_CONFIG['checkpoint_dir'],
)

# 逐个子集生成
all_results = {}
all_stats = []

for subset_name in SYNTHETIC_CONFIG['multipliers'].keys():
    if subset_name not in seed_data:
        print(f"跳过 {subset_name}: 种子数据中不存在")
        continue

    seeds = seed_data[subset_name]
    multiplier = SYNTHETIC_CONFIG['multipliers'][subset_name]
    strategy = SYNTHETIC_CONFIG['strategies'][subset_name]

    print(f"\n{'='*60}")
    print(f"处理子集: {subset_name}")
    print(f"策略: {strategy} | 倍数: {multiplier}")
    print(f"{'='*60}")

    try:
        results, stats = pipeline.generate_subset(
            subset_name=subset_name,
            seeds=seeds,
            strategy=strategy,
            multiplier=multiplier,
            batch_size=api_config['batch_size'],
        )

        all_results[subset_name] = results
        all_stats.append(stats)

        # 保存子集结果
        output_path = pipeline.save_subset(subset_name, results)
        print(f"已保存到: {output_path}")

    except Exception as e:
        print(f"错误: {subset_name} 生成失败: {e}")
        print("检查点已保存，修复问题后重新运行此单元格可从中断处继续")
        raise

print(f"\n{'='*60}")
print("所有子集生成完成!")
print(f"{'='*60}")

## 2b.5 生成统计

In [ ]:
# 显示生成统计
print("生成统计:")
print("-" * 60)

total_samples = 0
total_seeds = 0
total_synthetic = 0

for stats in all_stats:
    subset = stats['subset']
    print(f"{subset}:")
    print(f"  总样本: {stats['total_samples']}")
    print(f"  原始种子: {stats['seed_samples']}")
    print(f"  合成样本: {stats['synthetic_samples']}")
    if stats['total_samples'] > 0:
        real_pct = stats['seed_samples'] / stats['total_samples'] * 100
        print(f"  真实比例: {real_pct:.1f}%")
    total_samples += stats['total_samples']
    total_seeds += stats['seed_samples']
    total_synthetic += stats['synthetic_samples']

print("-" * 60)
print(f"总计:")
print(f"  总样本: {total_samples}")
print(f"  原始种子: {total_seeds}")
print(f"  合成样本: {total_synthetic}")
if total_samples > 0:
    print(f"  真实比例: {total_seeds/total_samples*100:.1f}%")

## 2b.6 质量关卡

三层过滤确保合成数据质量:
1. **Token长度过滤**: 移除超过 max_tokens 限制的样本 (防止训练时静默截断)
2. **困惑度过滤** (可选): 使用基座模型计算样本困惑度, 过滤高困惑度样本 (默认关闭, 需约20GB内存)
3. **多样性评分** (默认开启): 计算n-gram distinct-1/2指标, 自动去重低多样性样本 (纯文本处理, 无需模型)


In [ ]:
# 加载 tokenizer 用于长度检查
model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])
_, tokenizer = load_model_and_tokenizer(
    model_name_or_path=model_path,
    quantization_config=None,
    device_map='cpu',
)

max_tokens = SYNTHETIC_CONFIG['quality_gates']['max_tokens']

# 合并所有子集结果
combined_samples = []
for subset_name, results in all_results.items():
    combined_samples.extend(results)

print(f"质量关卡前总样本: {len(combined_samples)}")

# 过滤超过长度限制的样本
filtered_samples = []
removed_count = 0

for sample in combined_samples:
    # 构建对话文本用于长度检查
    text = f"{sample.get('instruction', '')}\n{sample.get('input', '')}\n{sample.get('output', '')}"
    tokens = tokenizer.encode(text, add_special_tokens=False)

    if len(tokens) <= max_tokens:
        filtered_samples.append(sample)
    else:
        removed_count += 1

print(f"过滤后样本: {len(filtered_samples)}")
print(f"移除样本: {removed_count} (超过 {max_tokens} tokens)")

# ==================== 质量关卡: 困惑度过滤 & 多样性评分 ====================
from utils.synthetic_pipeline import filter_by_perplexity, filter_by_diversity

qg_config = SYNTHETIC_CONFIG['quality_gates']

# --- 困惑度过滤 (可选, 默认关闭) ---
if qg_config.get('perplexity', {}).get('enabled', False):
    print("\n困惑度过滤已启用, 加载基座模型...")
    # 注意: 加载基座模型需要约20GB内存
    # 如果内存不足, 请在config.py中将 perplexity.enabled 设为 False
    _, ppl_model = load_model_and_tokenizer(
        model_name_or_path=model_path,
        quantization_config=None,
        device_map='cpu',
    )
    filtered_samples, ppl_threshold = filter_by_perplexity(
        filtered_samples,
        model=ppl_model,
        tokenizer=tokenizer,  # 复用已加载的tokenizer
        percentile=qg_config['perplexity']['percentile'],
        device=qg_config['perplexity'].get('device'),
    )
    print(f"困惑度过滤后: {len(filtered_samples)} 条样本 (阈值={ppl_threshold:.2f})")
    del ppl_model
    torch.cuda.empty_cache()
else:
    print("\n困惑度过滤: 已跳过 (默认关闭, 可在config.py中启用)")

# --- 多样性评分 (默认开启, 纯文本处理) ---
if qg_config.get('diversity', {}).get('enabled', True):
    print("\n多样性评分...")
    filtered_samples, div_stats = filter_by_diversity(
        filtered_samples,
        min_distinct_1=qg_config['diversity']['min_distinct_1'],
        min_distinct_2=qg_config['diversity']['min_distinct_2'],
        field=qg_config['diversity'].get('field', 'instruction'),
    )
    print(f"多样性评分: distinct-1={div_stats['distinct_1']}, distinct-2={div_stats['distinct_2']}")
    print(f"多样性过滤后: {len(filtered_samples)} 条样本")
else:
    print("\n多样性评分: 已跳过")


# 显示长度分布
lengths = []
for sample in filtered_samples:
    text = f"{sample.get('instruction', '')}\n{sample.get('input', '')}\n{sample.get('output', '')}"
    lengths.append(len(tokenizer.encode(text, add_special_tokens=False)))

import numpy as np
print(f"\n长度统计:")
print(f"  平均: {np.mean(lengths):.0f} tokens")
print(f"  中位数: {np.median(lengths):.0f} tokens")
print(f"  最大: {np.max(lengths)} tokens")
print(f"  最小: {np.min(lengths)} tokens")

# 清理显存
import torch
del tokenizer
torch.cuda.empty_cache()

## 2b.7 转换为 ChatML 格式

In [ ]:
# 将过滤后的样本按子集重新组织
filtered_by_subset = {}
for sample in filtered_samples:
    subset = sample.get('subset', 'unknown')
    if subset not in filtered_by_subset:
        filtered_by_subset[subset] = []
    filtered_by_subset[subset].append({
        'instruction': sample['instruction'],
        'input': sample.get('input', ''),
        'output': sample['output'],
    })

# 使用 convert_to_chatml 转换为对话格式
# 注意：convert_to_chatml 需要 tokenizer，我们在上一步已加载
# 这里重新加载（如果显存已清理）
_, tokenizer = load_model_and_tokenizer(
    model_name_or_path=model_path,
    quantization_config=None,
    device_map='cpu',
)

dataset = convert_to_chatml(
    data=filtered_by_subset,
    tokenizer=tokenizer,
    system_message=DATA_CONFIG['chatml']['system_message']
)

print(f"ChatML 转换完成:")
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data)} 条")

# 显示一个示例
sample = dataset['train'][0]
print(f"\n示例 (前 500 字符):")
print(sample['text'][:500])

# 清理
del tokenizer
torch.cuda.empty_cache()

## 2b.8 保存数据集并注册到流水线状态

In [ ]:
# 保存处理后的数据集
processed_dir = DATA_CONFIG['processed_data_dir']
save_dataset(dataset, processed_dir)

print(f"数据集已保存到: {processed_dir}")

# 同时保存原始合成数据（不含ChatML格式）用于参考
synthetic_dir = SYNTHETIC_CONFIG['output_dir']
combined_raw_path = Path(synthetic_dir) / "combined_raw.json"
with open(combined_raw_path, 'w', encoding='utf-8') as f:
    json.dump(filtered_by_subset, f, ensure_ascii=False, indent=2)
print(f"原始合成数据已保存到: {combined_raw_path}")

# 注册到流水线状态
state = PipelineState()

state.register(
    name="synthetic_raw",
    path=str(combined_raw_path),
    artifact_type="data",
    metadata={
        "total_samples": len(filtered_samples),
        "subsets": list(filtered_by_subset.keys()),
        "seed_count": total_seeds,
        "synthetic_count": total_synthetic,
    }
)

state.register(
    name="synthetic_dataset",
    path=processed_dir,
    artifact_type="dataset",
    metadata={
        "splits": {k: len(v) for k, v in dataset.items()},
        "format": "chatml",
        "total": sum(len(v) for v in dataset.values()),
    }
)

# 显示流水线状态摘要
summary = state.summary()
print(f"\n流水线状态摘要:")
print(f"  状态文件: {summary['state_file']}")
print(f"  总工件数: {summary['total_artifacts']}")
print(f"  工件列表: {summary['artifact_names']}")

---

## 完成

合成数据生成完成！数据集已保存并注册到流水线状态。

### 下一步

1. 检查 `data/processed/` 目录下的数据集文件
2. 打开 **03_model_benchmark.ipynb** 进行微调前基准测试
3. 或打开 **04_qlora_finetune.ipynb** 开始训练